# User Segmentation

Segment users by platform (iOS / Android), country (US / LatAm / EU), and acquisition source.
Compute D7 retention and ARPU90d per segment.

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='hopeful-list-429812-f3')

df = client.query('''
WITH base AS (
  SELECT
    user_id,
    JSON_VALUE(user_metadata, '$.platform') AS platform,
    country,
    JSON_VALUE(user_metadata, '$.utm_source') AS source
  FROM `hopeful-list-429812-f3.events.app_raw_table`
  WHERE event_name = 'app_install'
    AND DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 60 DAY)
  GROUP BY user_id, platform, country, source
)
SELECT platform, country, source, COUNT(*) AS users
FROM base GROUP BY platform, country, source
ORDER BY users DESC LIMIT 50
''').to_dataframe()
df